## Data Collection & Dataset Splitting

### Objectives
* Setup Kaggle authentication and download the official Cherry Leaves dataset from Kaggle.
* Inspect and clean dataset by removing non-image files.
* Split dataset randomly into **Train (70%)**, **Validation (10%)**, and **Test (20%)** sets.
* Plot and save the image distribution across splits to verify class balance.

### Inputs
* `kaggle.json` API authentication file located in project root.
* Kaggle Dataset: `codeinstitute/cherry-leaves`

### Outputs
* `inputs/cherry_leaves/` directory containing `train`, `validation`, and `test` folders split into `healthy` and `powdery_mildew`.
* `outputs/v1/class_distribution.png` visual plot.

In [1]:
import os
import shutil
import random
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# 1. Resolve Project Root Directory
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'jupyter_notebooks':
    PROJECT_DIR = os.path.abspath(os.path.join(current_dir, '..'))
else:
    PROJECT_DIR = current_dir

# 2. Configure Kaggle Credentials Path
os.environ['KAGGLE_CONFIG_DIR'] = PROJECT_DIR

# 3. Define Project Subdirectories
INPUTS_DIR = os.path.join(PROJECT_DIR, 'inputs')
RAW_DATA_DIR = os.path.join(INPUTS_DIR, 'raw_data')
SPLIT_DATA_DIR = os.path.join(INPUTS_DIR, 'cherry_leaves')
OUTPUTS_DIR = os.path.join(PROJECT_DIR, 'outputs', 'v1')

# 4. Create Base Directories
for directory in [RAW_DATA_DIR, SPLIT_DATA_DIR, OUTPUTS_DIR]:
    os.makedirs(directory, exist_ok=True)

print(f"Project Root: {PROJECT_DIR}")
print(f"Split Dataset Dir: {SPLIT_DATA_DIR}")
print(f"Outputs Dir: {OUTPUTS_DIR}")

Project Root: d:\books\code\code-institut\PROJECTS\cherry tree leaves
Split Dataset Dir: d:\books\code\code-institut\PROJECTS\cherry tree leaves\inputs\cherry_leaves
Outputs Dir: d:\books\code\code-institut\PROJECTS\cherry tree leaves\outputs\v1


In [ ]:
! pip install kaggle

In [4]:
# Verify kaggle.json existence before importing API
kaggle_json_path = os.path.join(PROJECT_DIR, 'kaggle.json')

if not os.path.exists(kaggle_json_path):
    raise FileNotFoundError(
        f"kaggle.json not found in root directory '{PROJECT_DIR}'. "
        f"Please place your Kaggle API token in the project root."
    )

import kaggle

dataset_name = "codeinstitute/cherry-leaves"
print(f"Downloading dataset '{dataset_name}'...")

kaggle.api.dataset_download_files(dataset_name, path=RAW_DATA_DIR, unzip=True)
print("Download and extraction complete.")

Dataset URL: https://www.kaggle.com/datasets/codeinstitute/cherry-leaves
Download and extraction complete.


In [5]:
valid_extensions = ('.png', '.jpg', '.jpeg')
removed_count = 0
corrupted_count = 0

for root, _, files in os.walk(RAW_DATA_DIR):
    for file in files:
        file_path = os.path.join(root, file)
        
        # Check file extension
        if not file.lower().endswith(valid_extensions):
            os.remove(file_path)
            removed_count += 1
            print(f"Removed non-image file: {file_path}")
            continue

        # Verify image integrity
        try:
            with Image.open(file_path) as img:
                img.verify()
        except Exception:
            os.remove(file_path)
            corrupted_count += 1
            print(f"Removed corrupted image file: {file_path}")

print(f"Data cleaning complete. Invalid files removed: {removed_count}, Corrupted removed: {corrupted_count}")

Data cleaning complete. Invalid files removed: 0, Corrupted removed: 0


In [6]:
# Set random seed for reproducibility
random.seed(42)

# Define split ratios and labels
train_ratio, val_ratio, test_ratio = 0.70, 0.10, 0.20
labels = ['healthy', 'powdery_mildew']
splits = ['train', 'validation', 'test']

# Clear existing split directory to ensure clean idempotency
if os.path.exists(SPLIT_DATA_DIR):
    shutil.rmtree(SPLIT_DATA_DIR)

for split in splits:
    for label in labels:
        os.makedirs(os.path.join(SPLIT_DATA_DIR, split, label), exist_ok=True)

summary_data = []

# Dynamically locate label directories and copy files
for label in labels:
    label_dir = None
    for root, dirs, files in os.walk(RAW_DATA_DIR):
        if os.path.basename(root).lower() == label.lower():
            label_dir = root
            break

    if not label_dir or not os.path.exists(label_dir):
        raise FileNotFoundError(
            f"Could not locate folder for '{label}' in '{RAW_DATA_DIR}'."
        )

    images = [f for f in os.listdir(label_dir) if f.lower().endswith(valid_extensions)]
    random.shuffle(images)

    total_imgs = len(images)
    train_count = int(total_imgs * train_ratio)
    val_count = int(total_imgs * val_ratio)

    train_imgs = images[:train_count]
    val_imgs = images[train_count:train_count + val_count]
    test_imgs = images[train_count + val_count:]

    # Copy files
    for img_list, split_name in [(train_imgs, 'train'), (val_imgs, 'validation'), (test_imgs, 'test')]:
        for img in img_list:
            shutil.copy(os.path.join(label_dir, img), os.path.join(SPLIT_DATA_DIR, split_name, label, img))

    print(f"Category '{label}': {total_imgs} total -> Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

    summary_data.extend([
        {'Split': 'Train', 'Label': label.replace('_', ' ').title(), 'Count': len(train_imgs)},
        {'Split': 'Validation', 'Label': label.replace('_', ' ').title(), 'Count': len(val_imgs)},
        {'Split': 'Test', 'Label': label.replace('_', ' ').title(), 'Count': len(test_imgs)}
    ])

# Print Summary DataFrame
df_summary = pd.DataFrame(summary_data)
print("\nDataset Distribution Summary:\n", df_summary)

# Plot and Save Distribution Plot
plt.figure(figsize=(8, 5))
sns.barplot(data=df_summary, x='Split', y='Count', hue='Label')
plt.title('Image Distribution Across Dataset Splits')
plt.ylabel('Number of Images')

plot_path = os.path.join(OUTPUTS_DIR, 'class_distribution.png')
plt.savefig(plot_path, bbox_inches='tight')
plt.close()
print(f"\nDistribution plot saved to: {plot_path}")

Category 'healthy': 2104 total -> Train: 1472, Val: 210, Test: 422
Category 'powdery_mildew': 2104 total -> Train: 1472, Val: 210, Test: 422

Dataset Distribution Summary:
         Split           Label  Count
0       Train         Healthy   1472
1  Validation         Healthy    210
2        Test         Healthy    422
3       Train  Powdery Mildew   1472
4  Validation  Powdery Mildew    210
5        Test  Powdery Mildew    422

Distribution plot saved to: d:\books\code\code-institut\PROJECTS\cherry tree leaves\outputs\v1\class_distribution.png
